In [32]:
from rdkit import Chem
from rdkit.Chem import Draw, AllChem, rdMolAlign
import pandas as pd
#reading all ligands for Micobacterium tuberculosis
ligands = pd.read_excel('Mtb.xlsx', sheet_name=1)
#Name SMILES IC50 of all aurachin D homologues
ligands_Qloop = ligands[ligands['Binding site'] == 'Q-Loop']
ligands_Qloop_needed_columns = ligands_Qloop[['Name', 'SMILES', 'IC50 μM']]
#taking 1/3 of the data as training set for a model
training_set = ligands_Qloop_needed_columns[ligands_Qloop_needed_columns['IC50 μM'] < 0.3]

print(training_set)


            Name                                             SMILES  IC50 μM
2     Aurachin D  CC1=C(C(=O)C2=CC=CC=C2N1)C/C=C(\C)/CC/C=C(\C)/...    0.150
3   CK-3-22 (1T)  Cc4c(c2ccc(Oc1ccc(OC(F)(F)F)cc1)nc2)[nH]c3cccc...    0.140
8        MTD-403     Cc4c(c2ccc(N1CCCCC1)cc2)[nH]c3cc(F)cc(F)c3c4=O    0.270
9        CK-2-88          Cc4c(c2ccc(Cc1ccccc1)cc2)[nH]c3ccccc3c4=O    0.020
11       CK-2-63  Cc4c(c2ccc(Oc1ccc(OC(F)(F)F)cc1)cc2)[nH]c3cccc...    0.003
12        PG-203  Cc2[nH]c1ccccc1c(=O)c2c4ccc(Oc3ccc(OC(F)(F)F)c...    0.070
15          LT-9        O=c3cc(c2ccc(Cc1ccc(F)cc1)cc2)[nH]c4ccccc34    0.100
16        GN-171  CCOC(=O)c4c(c2ccc(Cc1ccc(OC(F)(F)F)cc1)cc2)[nH...    0.250
18       SL-2-25  Cc4c(c2ccc(c1ccc(OC(F)(F)F)cc1)nc2)[nH]c3ccccc...    0.290
19     WDH-1U-10  CCOC(=O)c4c(c2ccc(c1ccc(Cl)cc1)cc2)[nH]c3ccccc...    0.012
22      WDH-2G-6  CC(C)c4c(c2cnn(Cc1ccc(OC(F)(F)F)cc1)c2)[nH]c3c...    0.082


In [33]:
def Selecting_All_Confirmations(SMILES):
    #Creating molecule and preparing for conformer generation
    mol = Chem.MolFromSmiles(SMILES)
    mol = Chem.AddHs(mol)
    AllChem.EmbedMolecule(mol, AllChem.ETKDG())
    AllChem.MMFFOptimizeMolecule(mol)
    params = AllChem.ETKDGv3()
    #parameters of conforment simulation
    params.numThreads = 0
    params.randomSeed = 42
    params.pruneRmsThresh = -1 
    #2000 confirmantions
    confs = AllChem.EmbedMultipleConfs(
        mol,
        numConfs=2000,
        params=params
    )
    #conformer optimization
    results = AllChem.MMFFOptimizeMoleculeConfs(
        mol,
        mmffVariant='MMFF94s',
        maxIters=500
    )
    #energy calculations
    energies = [(i, res[1]) for i, res in enumerate(results)]
    energies.sort(key=lambda x: x[1])
    #energy window for conformers
    Emin = energies[0][1]
    energy_window = 5.0

    lowE = [idx for idx, E in energies if E <= Emin + energy_window]
    #rmsd filtration
    selected = []
    rmsd_cutoff = 0.7

    for idx in lowE: 
        if all(
            rdMolAlign.GetBestRMS(
                mol, mol,
                prbId=idx,
                refId=s,
                symmetrizeConjugatedTerminalGroups=True
            ) > rmsd_cutoff
            for s in selected
        ):
            selected.append(idx)
#return mols with needed conformers
    new_mol = Chem.Mol(mol)
    new_mol.RemoveAllConformers()
    for idx in selected:
        new_mol.AddConformer(mol.GetConformer(idx), assignId=True)
        
    return new_mol


In [ ]:
training_set['molecule'] = training_set['SMILES'].apply(Selecting_All_Confirmations)